# Very Attentive Tacotron (VAT) — Training Notebook

Based on **arXiv:2410.22179** — *"Robust and Unbounded Length Generalization in Autoregressive Transformer-Based TTS"*

This notebook:
1. Reviews and fixes the VAT implementation in `model.py`
2. Provides a corrected, self-contained VAT class
3. Trains on LJ Speech (or a small synthetic dataset for quick testing)
4. Logs losses and alignment diagnostics

## 0. Install Dependencies

In [ ]:
# Run once if needed
# !pip install torch torchaudio librosa matplotlib tensorboard

## 1. Environment Setup

In [ ]:
import os, sys
from pathlib import Path

# Locate project root (the folder that contains commons/dataset.py)
ROOT = Path.cwd().resolve()
if ROOT.name in ("tacotron2", "very-attentive-tacotron"):
    ROOT = ROOT.parent
for _ in range(4):
    if (ROOT / "commons" / "dataset.py").exists():
        break
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

## 2. Download Omani Dataset

Downloads the `dataset_new_omani` corpus from Google Drive.  
Skipped automatically if the files are already present.

Expected layout after extraction:
```
dataset_new_omani/
├── transcriptions.xlsx
└── clean_flac/
    └── *.flac
```

In [ ]:
import subprocess, sys, zipfile, shutil
from pathlib import Path

# ── Google Drive file ID (same zip as omani_speaker_finetune.ipynb) ───────────
DATASET_ZIP_FILE_ID = "1nFMp2NmH4hkuz9Qc7NRjq6OSj7vMAGzQ"

DATASET_DEST = ROOT / "dataset_new_omani"
AUDIO_DIR    = DATASET_DEST / "clean_flac"
XLSX_DEST    = DATASET_DEST / "transcriptions.xlsx"

# ── Ensure gdown is available ─────────────────────────────────────────────────
try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "--root-user-action=ignore", "gdown"])
    import gdown

# ── Skip if already present ───────────────────────────────────────────────────
if AUDIO_DIR.exists() and any(AUDIO_DIR.glob("*.flac")) and XLSX_DEST.exists():
    n_flac = len(list(AUDIO_DIR.glob("*.flac")))
    print(f"Dataset already present ({n_flac} flac files) — skipping download.")
else:
    zip_path = DATASET_DEST / "dataset.zip"
    DATASET_DEST.mkdir(parents=True, exist_ok=True)
    AUDIO_DIR.mkdir(exist_ok=True)
    print("Downloading dataset zip from Google Drive ...")
    gdown.download(id=DATASET_ZIP_FILE_ID, output=str(zip_path), quiet=False, fuzzy=True)
    print(f"Extracting {zip_path.name} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.namelist():
            filename = Path(member).name
            if not filename:
                continue
            if member.lower().endswith(".flac"):
                with zf.open(member) as src, open(AUDIO_DIR / filename, "wb") as dst:
                    shutil.copyfileobj(src, dst)
            elif member.lower().endswith(".xlsx"):
                with zf.open(member) as src, open(XLSX_DEST, "wb") as dst:
                    shutil.copyfileobj(src, dst)
    zip_path.unlink()
    print(f"Extracted {len(list(AUDIO_DIR.glob('*.flac')))} .flac files → {AUDIO_DIR}")
    print(f"Extracted transcriptions.xlsx → {XLSX_DEST}")

print("\nDataset root:", DATASET_DEST)
DATASET_ROOT = str(DATASET_DEST)

## 1. Implementation Review — Paper vs. Code

Verified against arXiv:2410.22179 (HTML full text).

### Deviations fixed in `model.py`

| # | Component | Original deviation | Fix applied |
|---|-----------|-------------------|-------------|
| 1 | `InterpolatedRPB._bucket_index` | Used `torch.floor(eta)` — for negative η this rounds *away* from zero, but paper Eq. 4 uses ⌊η⌋₀ = sgn(η)·⌊\|η\|⌋ (round *toward* zero) | Decomposed into `sign * floor(abs_eta)` |
| 2 | `InterpolatedRPB` init | `randn * 0.02` for all biases | Cross-attention IRPBs now use paper's Gaussian window (σ=15, log-normalised); self-attention IRPBs use truncated-normal |
| 3 | `InterpolatedRPB` — self-attn buckets | 16 buckets used everywhere | Self-attention uses **32 buckets** (paper Sec. 3.3: "32 buckets for negative distances in causal decoder") |
| 4 | `AlignmentLayer` — softplus bias | No init — default 0, gives initial Δp ≈ ln(2) ≈ 0.69 | Bias initialised to **−1.25** (paper Sec. 3.2: gives initial avg Δp ≈ 0.25) |
| 5 | `VATCrossAttention` — MDP formula | `P_MD * clamp(|d|/D − 1, 0)` — off by factor D | Corrected to `P_MD * clamp(|d| − D, 0)` matching paper Eq. 7 |
| 6 | `VATDecoderLayer` — self-attention | Used `nn.MultiheadAttention` (no RPBs) | Now uses explicit IRPB-augmented self-attention with 32 buckets |
| 7 | Encoder | `TransformerEncoder` only (no conv) | Replaced with paper encoder: **2 conv stages** (stride-1 + stride-2) + **3 non-causal SA blocks** |
| 8 | Loss | MSE on raw mel frames | Architecture now outputs **VQ-code logits** for NLL training; mel-MSE head kept for prototyping |

### Architectural differences (by design)

| Aspect | Paper (reference config) | This implementation |
|--------|--------------------------|---------------------|
| Decoder width | 1024 | Configurable (`dec_embed_dim`, default 1024) |
| Decoder heads | 16 | Configurable (`dec_heads`, default 16) |
| Encoder width | 512 | Configurable (`enc_embed_dim`, default 512) |
| Output | VQ-VAE code NLL | VQ-code logits + mel-MSE fallback |
| Training steps | 650k Adam steps | User-defined |

## 2. Import Corrected VAT Model

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple

# Import from the corrected model.py in this directory
from model import (
    VeryAttentiveTacotron,
    VATEncoder,
    AlignmentLayer,
    VATCrossAttention,
    VATDecoderLayer,
    InterpolatedRPB,
)

print("VAT model imported from model.py")

## 3. Sanity Check — Forward Pass

## 4. Omani Dataset — Load & Inspect

Uses `NewOmaniTTSDataset` from `commons/dataset.py`:
- Reads `dataset_new_omani/transcriptions.xlsx` (columns: `File Name`, `Text`)
- Loads FLAC audio from `dataset_new_omani/clean_flac/`
- Pads 100 ms silence at each end (same as the Tacotron2 fine-tuning pipeline)
- Pre-computes `mel_lengths` for `BucketBatchSampler` without decoding audio
- Uses `TTSCollator` (the shared project collator) which returns:
  `(text_padded, input_lengths, mel_padded, gate_padded, enc_mask, dec_mask)`
  where `mel_padded` is `(B, T_mel, 80)` and masks are `True = padding`

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split

from commons.dataset     import NewOmaniTTSDataset, TTSCollator, BucketBatchSampler
from commons.hyperparams import Tacotron2Config
from tacotron2.tokenizer import Tokenizer

# ── Audio / mel config (use same values as the Tacotron2 pipeline) ────────────
config = Tacotron2Config()
tok    = Tokenizer()

# ── Dataset ───────────────────────────────────────────────────────────────────
# speakers=None  →  use all speakers in the corpus
# Reduce max_audio_seconds if GPU memory is tight
full_ds = NewOmaniTTSDataset(
    dataset_root      = DATASET_ROOT,
    speakers          = None,          # or e.g. ["111", "49"] to filter
    sample_rate       = config.sample_rate,
    n_fft             = config.n_fft,
    window_size       = config.win_length,
    hop_size          = config.hop_length,
    fmin              = config.fmin,
    fmax              = config.fmax,
    num_mels          = config.n_mels,
    min_db            = config.min_db,
    max_scaled_abs    = config.max_scaled_abs,
    max_audio_seconds = 10.0,
)

print(f"Total samples : {len(full_ds)}")
print(f"Speakers      : {full_ds.available_speakers}")

# ── Train / val split ─────────────────────────────────────────────────────────
VAL_SPLIT      = 0.1
VAL_SPLIT_SEED = 42
BATCH_SIZE     = 8

n_val   = max(1, int(len(full_ds) * VAL_SPLIT))
n_train = len(full_ds) - n_val
gen     = torch.Generator().manual_seed(VAL_SPLIT_SEED)
train_ds, val_ds = random_split(full_ds, [n_train, n_val], generator=gen)
print(f"Train: {n_train}   Val: {n_val}")

# BucketBatchSampler groups by mel length to minimise padding waste
train_mel_lengths = [full_ds.mel_lengths[i] for i in train_ds.indices]
train_sampler     = BucketBatchSampler(
    train_mel_lengths, batch_size=BATCH_SIZE, drop_last=True
)

collate_fn = TTSCollator()

train_dl = DataLoader(train_ds, batch_sampler=train_sampler,
                      collate_fn=collate_fn, num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                      collate_fn=collate_fn, num_workers=0)

# ── Inspect one batch ─────────────────────────────────────────────────────────
# TTSCollator returns: (text_pad, input_lengths, mel_pad, gate_pad, enc_mask, dec_mask)
text_pad, in_lens, mel_pad, gate_pad, enc_mask, dec_mask = next(iter(train_dl))
print(f"\nBatch shapes:")
print(f"  text_pad   : {text_pad.shape}   (B, T_text)")
print(f"  in_lens    : {in_lens.tolist()}")
print(f"  mel_pad    : {mel_pad.shape}   (B, T_mel, 80)")
print(f"  gate_pad   : {gate_pad.shape}  (B, T_mel)  — 1.0 at last real frame")
print(f"  enc_mask   : {enc_mask.shape}  (B, T_text) — True=padding")
print(f"  dec_mask   : {dec_mask.shape}  (B, T_mel)  — True=padding")

# Mel spectrogram of first sample
fig, ax = plt.subplots(figsize=(12, 3))
ax.imshow(mel_pad[0].T.cpu().numpy(), aspect="auto", origin="lower", cmap="viridis")
ax.set_title("Sample mel spectrogram (first item in batch)")
ax.set_xlabel("Frame"); ax.set_ylabel("Mel bin")
plt.tight_layout(); plt.show()

# Mel-length distribution
mel_lens = [full_ds.mel_lengths[i] for i in range(len(full_ds))]
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(mel_lens, bins=40, color="steelblue", edgecolor="white")
ax.set_xlabel("Mel frames"); ax.set_ylabel("Count")
ax.set_title("Mel length distribution — Omani dataset")
plt.tight_layout(); plt.show()

print(f"\nVocab size: {tok.vocab_size}  (use this for VOCAB_SIZE below)")

## 5. Loss Functions

In [ ]:
class VATLoss(nn.Module):
    """
    Combined VAT loss:
    - L_mel:   MSE between predicted and target mel (masked)
    - L_stop:  Binary cross-entropy on stop token (weighted for class imbalance)
    - L_align: Monotonicity regularisation — penalise large Δpos increments
               that would skip over encoder tokens
    """
    def __init__(self, mel_weight=1.0, stop_weight=5.0, align_weight=0.1):
        super().__init__()
        self.mel_w   = mel_weight
        self.stop_w  = stop_weight
        self.align_w = align_weight

    def forward(
        self,
        mel_pred:   torch.Tensor,   # (B, T, n_mels)
        mel_target: torch.Tensor,   # (B, T, n_mels)
        stop_pred:  torch.Tensor,   # (B, T, 1)  logits
        stop_target:torch.Tensor,   # (B, T)
        align_pos:  torch.Tensor,   # (B, T)
        mel_mask:   torch.Tensor,   # (B, T)  True = padding
    ) -> Tuple[torch.Tensor, dict]:
        # --- Mel loss (ignore padded frames) --------------------------------
        valid = ~mel_mask                                  # (B, T)
        mel_diff = (mel_pred - mel_target).pow(2)         # (B, T, n_mels)
        mel_loss = mel_diff[valid].mean()

        # --- Stop loss ------------------------------------------------------
        stop_pred_flat  = stop_pred.squeeze(-1)[valid]    # (N,)
        stop_target_flat = stop_target[valid]             # (N,)
        # Positive weight to handle class imbalance (most frames are not stop)
        pos_w = (stop_target_flat == 0).sum().float() / (stop_target_flat == 1).sum().float().clamp(min=1)
        stop_loss = F.binary_cross_entropy_with_logits(
            stop_pred_flat, stop_target_flat,
            pos_weight=torch.tensor(pos_w, device=stop_pred.device)
        )

        # --- Alignment regularisation (penalise Δpos > 1 to avoid skipping) -
        delta = (align_pos[:, 1:] - align_pos[:, :-1]).clamp(min=0)  # monotonic by construction
        align_loss = (delta - 1.0).pow(2).mean()   # encourage Δ≈1 (steady progression)

        total = self.mel_w * mel_loss + self.stop_w * stop_loss + self.align_w * align_loss
        losses = {
            'total': total.item(), 'mel': mel_loss.item(),
            'stop': stop_loss.item(), 'align': align_loss.item()
        }
        return total, losses


criterion = VATLoss()
print("Loss function ready.")

In [ ]:
import math, torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ── Model config ──────────────────────────────────────────────────────────────
# vocab_size=113: matches Tokenizer (Arabic + English + diacritics + special tokens)
# n_mels=80:      matches Tacotron2Config / NewOmaniTTSDataset pipeline
# Paper reference config shown in comments.
VOCAB_SIZE    = tok.vocab_size   # 113
N_MELS        = config.n_mels   # 80
ENC_EMBED_DIM = 256              # paper: 512
ENC_HEADS     = 4                # paper: 8
DEC_EMBED_DIM = 256              # paper: 1024
DEC_HEADS     = 4                # paper: 16
DEC_LAYERS    = 4                # paper: 6
D_KV          = 64
RNN_HIDDEN    = 128              # paper: 256
DROPOUT       = 0.1

# ── Training ──────────────────────────────────────────────────────────────────
FINETUNE_EPOCHS = 300
LR              = 1e-4    # fine-tune at lower LR (matches omani_speaker_finetune)
WARMUP          = 1000    # steps
GRAD_CLIP       = 1.0
LOG_EVERY       = 20
SAVE_EVERY      = 30      # save checkpoint every N epochs
CKPT_DIR        = ROOT / "checkpoints_vat_omani"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Build model ───────────────────────────────────────────────────────────────
from model import VeryAttentiveTacotron

vat_model = VeryAttentiveTacotron(
    vocab_size      = VOCAB_SIZE,
    enc_embed_dim   = ENC_EMBED_DIM,
    enc_heads       = ENC_HEADS,
    dec_embed_dim   = DEC_EMBED_DIM,
    dec_heads       = DEC_HEADS,
    dec_layers      = DEC_LAYERS,
    d_kv            = D_KV,
    n_mels          = N_MELS,
    vq_codebook_size= 1024,
    vq_num_codebooks= 8,
    rnn_hidden      = RNN_HIDDEN,
    dropout         = DROPOUT,
).to(device)

n_params = sum(p.numel() for p in vat_model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,}")
print(f"Vocab size : {VOCAB_SIZE}")
print(f"n_mels     : {N_MELS}")

optimizer = torch.optim.AdamW(vat_model.parameters(), lr=LR, weight_decay=1e-2)

total_steps = FINETUNE_EPOCHS * len(train_dl)
def lr_lambda(step):
    step = max(step, 1)
    if step < WARMUP:
        return step / WARMUP
    return max(0.05, 0.5 * (1 + math.cos(math.pi * (step - WARMUP) / (total_steps - WARMUP))))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
print(f"Total steps: {total_steps:,}  (epochs={FINETUNE_EPOCHS}, batches/epoch={len(train_dl)})")

## 7. Loss Function

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple


class VATLoss(nn.Module):
    """
    VAT training loss on raw mel frames (prototype mode, no VQ-VAE required).

    Components:
      L_mel   — MSE between predicted and target mel, masked on padding frames
      L_stop  — BCE on stop token (gate), pos_weight to handle class imbalance
      L_align — Penalises |Δp - 1| to encourage steady one-token-per-step progress
    """
    def __init__(self, mel_weight: float = 1.0,
                 stop_weight: float = 5.0,
                 align_weight: float = 0.1):
        super().__init__()
        self.mel_w   = mel_weight
        self.stop_w  = stop_weight
        self.align_w = align_weight

    def forward(
        self,
        mel_pred:    torch.Tensor,   # (B, T, 80)
        mel_target:  torch.Tensor,   # (B, T, 80)
        stop_pred:   torch.Tensor,   # (B, T, 1)  logits
        stop_target: torch.Tensor,   # (B, T)     1.0 at last real frame
        align_pos:   torch.Tensor,   # (B, T)
        mel_mask:    torch.Tensor,   # (B, T)     True = padding
    ) -> Tuple[torch.Tensor, dict]:
        valid = ~mel_mask   # (B, T)

        # Mel MSE on non-padded frames only
        mel_loss = (mel_pred - mel_target).pow(2)[valid].mean()

        # Stop BCE with positive-class reweighting
        sp = stop_pred.squeeze(-1)[valid]
        st = stop_target[valid]
        n_neg = (st == 0).sum().float()
        n_pos = (st == 1).sum().float().clamp(min=1)
        stop_loss = F.binary_cross_entropy_with_logits(
            sp, st, pos_weight=torch.tensor(n_neg / n_pos, device=sp.device)
        )

        # Alignment regularisor: penalise deviation from Δp=1
        delta     = align_pos[:, 1:] - align_pos[:, :-1]   # monotonic by Softplus
        align_loss = (delta - 1.0).pow(2).mean()

        total  = self.mel_w * mel_loss + self.stop_w * stop_loss + self.align_w * align_loss
        losses = dict(total=total.item(), mel=mel_loss.item(),
                      stop=stop_loss.item(), align=align_loss.item())
        return total, losses


criterion = VATLoss().to(device)
print("Loss function ready.")

## 8. Training Loop

`TTSCollator` batch format: `(text_pad, input_lengths, mel_pad, gate_pad, enc_mask, dec_mask)`
- `mel_pad`  — `(B, T_mel, 80)`, already transposed by the collator
- `gate_pad` — `(B, T_mel)`, 1.0 at the last real frame (stop target)
- `enc_mask` — `(B, T_text)`, `True` = padding  →  passed as `src_key_padding_mask`
- `dec_mask` — `(B, T_mel)`,  `True` = padding  →  used as `mel_mask` in the loss

In [ ]:
import time
from collections import defaultdict
from pathlib import Path


def run_epoch(model, loader, train_mel_lengths, optimizer, scheduler,
              criterion, device, train=True):
    model.train(train)
    totals    = defaultdict(float)
    n_batches = 0

    # Re-shuffle BucketBatchSampler buckets at the start of each training epoch
    if train and hasattr(loader, 'batch_sampler'):
        bs = loader.batch_sampler
        if isinstance(bs, BucketBatchSampler):
            bs.__init__(train_mel_lengths, batch_size=bs.batch_size,
                        drop_last=bs.drop_last)

    with torch.set_grad_enabled(train):
        for batch_idx, batch in enumerate(loader):
            # TTSCollator returns 6 values
            text_pad, in_lens, mel_pad, gate_pad, enc_mask, dec_mask = batch
            text_pad = text_pad.to(device)
            mel_pad  = mel_pad.to(device)    # (B, T_mel, 80)
            gate_pad = gate_pad.to(device)   # (B, T_mel)  stop target
            enc_mask = enc_mask.to(device)   # (B, T_text) True=pad
            dec_mask = dec_mask.to(device)   # (B, T_mel)  True=pad

            # forward_mel: takes (text, mels, src_key_padding_mask)
            mel_pred, stop_pred, align_pos = model.forward_mel(
                text_pad, mel_pad, enc_mask
            )

            loss, losses = criterion(
                mel_pred, mel_pad, stop_pred, gate_pad, align_pos, dec_mask
            )

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()

            for k, v in losses.items():
                totals[k] += v
            n_batches += 1

            if train and batch_idx % LOG_EVERY == 0:
                lr = optimizer.param_groups[0]['lr']
                print(f"  step {batch_idx:4d}/{len(loader)} | "
                      f"total {losses['total']:.4f} | mel {losses['mel']:.4f} | "
                      f"stop {losses['stop']:.4f} | align {losses['align']:.4f} | "
                      f"lr {lr:.2e}")

    return {k: v / max(n_batches, 1) for k, v in totals.items()}


# ── Training run ──────────────────────────────────────────────────────────────
history  = defaultdict(list)
best_val = float('inf')
last_ckpt_path = None

for epoch in range(1, FINETUNE_EPOCHS + 1):
    t0 = time.time()

    train_losses = run_epoch(vat_model, train_dl, train_mel_lengths,
                             optimizer, scheduler, criterion, device, train=True)
    val_losses   = run_epoch(vat_model, val_dl,   train_mel_lengths,
                             optimizer, scheduler, criterion, device, train=False)

    for k, v in train_losses.items():
        history[f'train_{k}'].append(v)
    for k, v in val_losses.items():
        history[f'val_{k}'].append(v)

    elapsed = time.time() - t0
    print(f"Epoch {epoch:3d}/{FINETUNE_EPOCHS} | "
          f"train {train_losses['total']:.4f}  val {val_losses['total']:.4f}  "
          f"[{elapsed:.1f}s]")

    # ── Checkpoint logic (mirrors omani_speaker_finetune) ─────────────────────
    is_milestone = (epoch % SAVE_EVERY == 0) or (epoch == FINETUNE_EPOCHS)
    ckpt_name    = f"vat_omani_epoch_{epoch:04d}.pth" if is_milestone \
                   else "vat_omani_last.pth"
    new_path     = CKPT_DIR / ckpt_name

    torch.save({
        'epoch'        : epoch,
        'model'        : vat_model.state_dict(),
        'optimizer'    : optimizer.state_dict(),
        'train_losses' : history['train_total'],
        'val_losses'   : history['val_total'],
        'config'       : dict(
            vocab_size=VOCAB_SIZE, enc_embed_dim=ENC_EMBED_DIM,
            dec_embed_dim=DEC_EMBED_DIM, dec_heads=DEC_HEADS,
            dec_layers=DEC_LAYERS, n_mels=N_MELS,
        ),
    }, new_path)

    # Delete previous non-milestone checkpoint to save disk space
    if last_ckpt_path and last_ckpt_path.exists():
        stem = last_ckpt_path.stem
        if '_epoch_' in stem:
            prev_ep = int(stem.split('_epoch_')[-1])
            if prev_ep % SAVE_EVERY != 0:
                last_ckpt_path.unlink(missing_ok=True)
        else:
            last_ckpt_path.unlink(missing_ok=True)
    last_ckpt_path = new_path

    if val_losses['total'] < best_val:
        best_val = val_losses['total']
        best_path = CKPT_DIR / "vat_omani_best.pth"
        torch.save(torch.load(new_path, map_location='cpu'), best_path)
        print(f"  ✓ New best val={best_val:.4f} → {best_path.name}")

print(f"\nTraining complete. Best val loss: {best_val:.4f}")
print(f"Checkpoints in: {CKPT_DIR}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key in zip(axes, ['total', 'mel', 'stop']):
    ax.plot(history[f'train_{key}'], label='train')
    ax.plot(history[f'val_{key}'],   label='val')
    ax.set_title(f'{key} loss')
    ax.set_xlabel('epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('vat_training_curves.png', dpi=150)
plt.show()

## 9. Alignment Visualisation

In [ ]:
vat_model.eval()
with torch.no_grad():
    # TTSCollator returns 6-tuple: (text_pad, in_lens, mel_pad, gate_pad, enc_mask, dec_mask)
    text_b, in_lens_b, mel_b, gate_b, enc_mask_b, dec_mask_b = next(iter(val_dl))
    text_b     = text_b.to(device)
    mel_b      = mel_b.to(device)
    enc_mask_b = enc_mask_b.to(device)
    _, _, align_b = vat_model.forward_mel(text_b, mel_b, enc_mask_b)

# Plot alignment trajectory for first sample in batch
ap    = align_b[0].cpu().numpy()
T_mel = ap.shape[0]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(T_mel), ap, color='steelblue', linewidth=1.5)
ax.set_xlabel('Decoder step (mel frame)')
ax.set_ylabel('Encoder position (align_pos)')
ax.set_title('VAT Alignment Trajectory (should be monotonically increasing)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('vat_alignment.png', dpi=150)
plt.show()

diffs = np.diff(ap)
print(f"Min Δ = {diffs.min():.4f}, Max Δ = {diffs.max():.4f}")
print(f"Monotonic: {(diffs >= 0).all()}")

## 10. IRPB Bucket Bias Visualisation

In [ ]:
# Show the learned IRPB biases from the first cross-attention layer
first_irpb = vat_model.decoder_layers[0].cross_attn.irpb
biases = first_irpb.bias.detach().cpu()  # (2*num_buckets, num_heads)

nb         = first_irpb.num_buckets
bucket_idx = torch.arange(-nb, nb).float()

fig, ax = plt.subplots(figsize=(10, 4))
for h in range(biases.shape[1]):
    ax.plot(bucket_idx.numpy(), biases[:, h].numpy(), alpha=0.7, label=f'head {h}')
ax.axvline(0, color='k', linewidth=0.8, linestyle='--')
ax.set_xlabel('Relative position bucket')
ax.set_ylabel('Bias value')
ax.set_title('Learned IRPB biases (layer 0 cross-attention)')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('vat_irpb_biases.png', dpi=150)
plt.show()

## 11. Quick Inference Test

In [ ]:
import matplotlib.pyplot as plt

# Load best checkpoint
best_ckpt_path = CKPT_DIR / "vat_omani_best.pth"
ckpt = torch.load(best_ckpt_path, map_location=device)
vat_model.load_state_dict(ckpt['model'])
print(f"Loaded checkpoint from epoch {ckpt['epoch']}")

# Tokenise a short Arabic test phrase
test_phrase = "مرحبا بالعالم"
test_tokens = torch.tensor(tok.encode(test_phrase),
                           dtype=torch.long, device=device).unsqueeze(0)  # (1, T_text)

# Autoregressive inference
vat_model.eval()
with torch.no_grad():
    mel_infer, align_infer = vat_model.inference(
        test_tokens, max_steps=500, stop_threshold=0.5
    )

print(f"Inferred mel shape : {mel_infer.shape}")
print(f"Alignment range    : [{align_infer.min():.2f}, {align_infer.max():.2f}]")

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].imshow(mel_infer.cpu().numpy().T, aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title('Inferred Mel Spectrogram')
axes[0].set_xlabel('Frame')
axes[0].set_ylabel('Mel bin')
axes[1].plot(align_infer.cpu().numpy())
axes[1].set_title('Inference Alignment Trajectory')
axes[1].set_xlabel('Decoder step')
axes[1].set_ylabel('Encoder position')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('vat_inference.png', dpi=150)
plt.show()

## 12. Replacing the Synthetic Dataset with LJ Speech

```python
# pip install torchaudio
import torchaudio
import torchaudio.transforms as T

LJ_SPEECH_ROOT = '/path/to/LJSpeech-1.1'

class LJSpeechDataset(Dataset):
    def __init__(self, root, split='train', n_mels=80, sample_rate=22050,
                 hop_length=256, win_length=1024, n_fft=1024, char2idx=None):
        self.dataset = torchaudio.datasets.LJSPEECH(root, download=True)
        self.mel_transform = T.MelSpectrogram(
            sample_rate=sample_rate, n_fft=n_fft, win_length=win_length,
            hop_length=hop_length, n_mels=n_mels
        )
        self.char2idx = char2idx or self._build_vocab()

    def _build_vocab(self):
        chars = sorted(set("".join(item[3] for item in self.dataset)))
        return {c: i+1 for i, c in enumerate(chars)}  # 0 reserved for padding

    def __len__(self): return len(self.dataset)

    def __getitem__(self, idx):
        waveform, sr, _, text = self.dataset[idx]
        mel = self.mel_transform(waveform).squeeze(0).log1p().T  # (T_mel, n_mels)
        tokens = torch.tensor([self.char2idx.get(c, 1) for c in text.lower()],
                               dtype=torch.long)
        stop = torch.zeros(mel.shape[0])
        stop[-1] = 1.0
        return tokens, mel, stop
```

Then replace `SyntheticTTSDataset` with `LJSpeechDataset` and update `VOCAB_SIZE` to `len(dataset.char2idx) + 1`.